# 06 — Misclassification Analysis

Deeper error analysis beyond the standard confusion matrix:
- Which classes are confused with which?
- Hardest examples (model was confidently wrong)
- Easiest mistakes (model was barely wrong)
- Per-class precision/recall ranked

## 1. Environment setup

This notebook is designed for **Google Colab (A100)**. The setup cell below:

1. Mounts Google Drive
2. Clones or updates the GitHub repo
3. Installs requirements
4. Adds the repo root to `sys.path` so `from src...` works

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
REPO_DIR = Path("/content/Image-Classification-with-CNN")

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping drive mount.")

# 2. Clone or pull
if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# 3. Make repo importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install requirements (Colab usually has torch already)
req = REPO_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

## 2. Pick a model and load it

In [ ]:
MODEL_NAME = "resnet50"      # or "efficientnet_b2"
BATCH_SIZE = 32

import torch, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from config.config import (
    CLASS_NAMES, DATASET_DIR, CHECKPOINTS_DRIVE,
    LOCAL_PLOTS, REPORT_FIGURES, LOCAL_METRICS,
)
from src.utils import get_device, set_seed
from src.dataset import build_dataloaders
from src.models import build_model
from src.train import load_model_for_inference
from src.evaluate import predict_on_loader, apply_ieee_style

set_seed(42)
device = get_device()
_, _, test_loader, _ = build_dataloaders(DATASET_DIR, batch_size=BATCH_SIZE)

model, _ = build_model(MODEL_NAME)
ckpt = CHECKPOINTS_DRIVE / f"{MODEL_NAME}_best.pth"
model = load_model_for_inference(model, ckpt, device)

## 3. Predictions on test set

In [ ]:
y_true, y_pred, y_prob = predict_on_loader(model, test_loader, device)
wrong = np.where(y_true != y_pred)[0]
print(f"Test images: {len(y_true)}")
print(f"Misclassified: {len(wrong)} ({100*len(wrong)/len(y_true):.1f}%)")

## 4. Confusion patterns — which pairs are confused most?

In [ ]:
from collections import Counter
pairs = Counter()
for i in wrong:
    t = CLASS_NAMES[y_true[i]]
    p = CLASS_NAMES[y_pred[i]]
    pairs[(t, p)] += 1

print("Top confused (true → predicted):")
for (t, p), n in pairs.most_common(10):
    print(f"  {t:<28s} → {p:<28s}  ({n} times)")

## 5. Most confidently wrong (model "sure" but wrong)

In [ ]:
if len(wrong):
    conf_wrong = y_prob[wrong, y_pred[wrong]]
    order = np.argsort(-conf_wrong)
    k = min(8, len(order))
    sel = wrong[order[:k]]

    apply_ieee_style()
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, idx in zip(axes.flat, sel):
        path, _ = test_loader.dataset.samples[idx]
        ax.imshow(Image.open(path).convert("RGB"))
        ax.axis("off")
        ax.set_title(
            f"pred: {CLASS_NAMES[y_pred[idx]]} ({y_prob[idx, y_pred[idx]]*100:.1f}%)\n"
            f"true: {CLASS_NAMES[y_true[idx]]}",
            fontsize=9,
        )
    fig.suptitle(f"{MODEL_NAME}: most confidently wrong predictions")
    fig.tight_layout()
    from src.utils import ensure_dir
    for d in (LOCAL_PLOTS, REPORT_FIGURES):
        ensure_dir(d)
        fig.savefig(d / f"{MODEL_NAME}_confidently_wrong.png", dpi=300, bbox_inches="tight")
        fig.savefig(d / f"{MODEL_NAME}_confidently_wrong.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No misclassifications — nothing to plot.")

## 6. Barely-wrong predictions (close to a correct call)

In [ ]:
if len(wrong):
    # Margin = prob(predicted) - prob(true). Closer to 0 = barely wrong.
    margin = y_prob[wrong, y_pred[wrong]] - y_prob[wrong, y_true[wrong]]
    order = np.argsort(margin)   # smallest margin first
    k = min(8, len(order))
    sel = wrong[order[:k]]
    print("Barely-wrong (true_prob nearly beat pred_prob):")
    for idx in sel:
        p_true = y_prob[idx, y_true[idx]]
        p_pred = y_prob[idx, y_pred[idx]]
        print(f"  true {CLASS_NAMES[y_true[idx]]:<28s} "
              f"p_true={p_true:.3f}   "
              f"pred {CLASS_NAMES[y_pred[idx]]:<28s} "
              f"p_pred={p_pred:.3f}   "
              f"margin={p_pred - p_true:.3f}")

## 7. Per-class accuracy ranked

In [ ]:
per_class_acc = []
for ci, cn in enumerate(CLASS_NAMES):
    idx = np.where(y_true == ci)[0]
    if len(idx) == 0:
        acc = float("nan")
    else:
        acc = float((y_pred[idx] == ci).mean())
    per_class_acc.append((cn, acc, len(idx)))

per_class_acc.sort(key=lambda x: x[1])
print(f"{'class':<32s} {'acc':>6s} {'n_test':>6s}")
for cn, a, n in per_class_acc:
    print(f"{cn:<32s} {a:>6.3f} {n:>6d}")

## 8. Save analysis summary

In [ ]:
import json
from src.utils import write_json
summary = {
    "model": MODEL_NAME,
    "n_test": int(len(y_true)),
    "n_wrong": int(len(wrong)),
    "test_accuracy": float((y_true == y_pred).mean()),
    "top_confused_pairs": [
        {"true": t, "pred": p, "count": n}
        for (t, p), n in pairs.most_common(20)
    ],
    "per_class_accuracy": [
        {"class": cn, "accuracy": float(a) if a == a else None, "support": int(n)}
        for cn, a, n in per_class_acc
    ],
}
write_json(summary, LOCAL_METRICS / f"{MODEL_NAME}_misclassification_summary.json")
print(f"saved: {LOCAL_METRICS}/{MODEL_NAME}_misclassification_summary.json")